[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/04_layernorm.ipynb)

# 🟡 Medium: Implement LayerNorm

Implement **Layer Normalization** from scratch.

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where $\mu$ and $\sigma^2$ are computed over the **last dimension**.

### Signature
```python
def my_layer_norm(
    x: torch.Tensor,      # input
    gamma: torch.Tensor,   # scale (same size as last dim)
    beta: torch.Tensor,    # shift (same size as last dim)
    eps: float = 1e-5
) -> torch.Tensor:
    ...
```

### Rules
- Do **NOT** use `F.layer_norm` or `torch.nn.LayerNorm`
- Normalize over the last dimension only
- Must support autograd

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.8 MB/s eta 0:00:00


In [2]:
import torch

In [45]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_layer_norm(x, gamma, beta, eps=1e-5):
  x_avg = torch.mean(x, dim=-1, keepdim=True)
  # Learning: correction=0 is needed, otherwise PyTorch would do Bessel correction (1/N-1 rather than 1/N)
  x_std = torch.std(x, dim=-1, keepdim=True, correction=0)
  return gamma * ((x - x_avg) / torch.sqrt(torch.pow(x_std, 2) + eps)) + beta


In [46]:
# 🧪 Debug
x = torch.randn(2, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)

print("X: ", x)

out = my_layer_norm(x, gamma, beta)
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)

print("Out: ", out)
print("Ref: ", ref)

print("Your output mean:", out.mean(dim=-1))   # should be ~0
print("Your output std: ", out.std(dim=-1))     # should be ~1
print("Match ref?      ", torch.allclose(out, ref, atol=1e-4))

X:  tensor([[-4.2581e-02,  6.4387e-04,  3.0722e-03, -1.2825e+00,  1.1786e+00,
         -9.1915e-01, -1.0339e+00,  3.9293e-01],
        [-9.4005e-01, -8.4392e-01,  2.0577e-01,  1.4979e+00, -1.9562e-01,
         -9.0611e-01,  2.5029e-01,  9.6346e-01]])
X_avg shape:  torch.Size([2, 1])
X_avg:  tensor([[-0.2128],
        [ 0.0040]])
X_std shape:  torch.Size([2, 1])
X_std:  tensor([[0.7699],
        [0.8465]])
Out:  tensor([[ 0.2212,  0.2773,  0.2805, -1.3894,  1.8074, -0.9174, -1.0664,  0.7869],
        [-1.1152, -1.0017,  0.2384,  1.7649, -0.2358, -1.0751,  0.2910,  1.1335]])
Ref:  tensor([[ 0.2212,  0.2773,  0.2805, -1.3894,  1.8074, -0.9174, -1.0664,  0.7869],
        [-1.1152, -1.0017,  0.2384,  1.7649, -0.2358, -1.0751,  0.2910,  1.1335]])
Your output mean: tensor([1.4901e-08, 0.0000e+00])
Your output std:  tensor([1.0690, 1.0690])
Match ref?       True


In [47]:
# ✅ SUBMIT
from torch_judge import check
check("layernorm")


🧪 Testing: Implement LayerNorm (Medium)
──────────────────────────────────────────────────
X_avg shape:  torch.Size([2, 3, 1])
X_avg:  tensor([[[-0.5048],
         [-0.2505],
         [-0.1885]],

        [[ 0.5556],
         [-0.6266],
         [-0.8609]]])
X_std shape:  torch.Size([2, 3, 1])
X_std:  tensor([[[0.8026],
         [1.0831],
         [0.6011]],

        [[0.8852],
         [1.0876],
         [0.8131]]])
  ✅ [1/3] Shape and basic behavior (2.3ms)
X_avg shape:  torch.Size([4, 1])
X_avg:  tensor([[-0.3215],
        [ 0.3518],
        [ 0.0662],
        [ 0.5469]])
X_std shape:  torch.Size([4, 1])
X_std:  tensor([[0.7596],
        [0.8504],
        [1.0641],
        [1.2810]])
  ✅ [2/3] With learned parameters (2.1ms)
X_avg shape:  torch.Size([2, 1])
X_avg:  tensor([[0.2817],
        [0.3846]], grad_fn=<MeanBackward1>)
X_std shape:  torch.Size([2, 1])
X_std:  tensor([[0.9669],
        [1.2067]], grad_fn=<StdBackward0>)
  ✅ [3/3] Gradient flow (3.0ms)
────────────────────────